# Lightweight CNN vs Vision Transformer for Plant Disease Classification on Edge Devices
**Comparative benchmark: MobileNetV3, EfficientNet-Lite, MobileViT on PlantVillage**

This notebook reproduces every experiment referenced in the paper draft:
1. Data preparation (PlantVillage, 38 classes, ~54k images)
2. Training eight architectures under an identical protocol (7 edge candidates + 1 non-lightweight reference)
3. Accuracy / F1 / confusion-matrix evaluation
4. Efficiency profiling: parameter count, FLOPs, model size, CPU latency (Raspberry-Pi proxy), INT8 quantization
5. Statistical testing: bootstrap confidence intervals, McNemar's paired test
6. Accuracy-vs-efficiency trade-off plots and export of all paper tables as CSV
7. **Robustness sweep** (blur / noise / brightness perturbations)
8. **Input-resolution ablation** (96 / 160 / 224 px)
9. **Grad-CAM** sanity check (is the model looking at the lesion or a shortcut?)
10. **Peak inference memory footprint**
11. **Part 2: PlantDoc** — identical protocol retrained/evaluated on a second, harder, real-world
    dataset (2,598 field-condition images, 27 classes), plus a cross-dataset comparison showing whether
    the PlantVillage architecture ranking survives domain shift
12. **Multi-seed retraining** — PlantVillage headline models (GhostNet-100, MobileViT-S) and all 8
    PlantDoc models across 3 seeds, plus a family-level (hybrid vs. CNN) significance test
13. **Cross-dataset transfer** — PlantVillage-trained models applied *without fine-tuning* to PlantDoc images over the classes the two datasets share, to separate domain shift from small-dataset effects
14. **Full static INT8 quantization via ONNX Runtime** — weights *and* activations quantized; accuracy, size and latency measured with the same runtime for FP32 vs INT8
15. Export everything (both parts, all extras) as a single zip

**Total runtime**: Part 1 (PlantVillage, 8 models) ≈ 6–8 hours; Part 2 (PlantDoc, 8 models, much
smaller dataset) adds roughly 1–1.5 hours.

**How to run:** Runtime → Change runtime type → GPU (T4). Run cells top to bottom.
Each model trains in ~40–70 min on a free Colab T4 GPU (15 epochs, 54k images). Total ≈ 6–8 hours for all eight models — this is a genuinely long run; the Drive-backed resumability below is what makes it survive multiple free-tier Colab sessions.

**Session-loss protection:** Section 1 mounts your Google Drive, and every result — checkpoints, best
weights, training history, tables, figures — is written there as it's produced, not to `/content`
(which is wiped when a Colab session recycles). If your free-tier session disconnects mid-run, just
reopen the notebook and re-run from the top: completed models are skipped, and an interrupted model
resumes from its last finished epoch instead of restarting.

> Replace the `KAGGLE_USERNAME` / `KAGGLE_KEY` cell with your own Kaggle API credentials (Kaggle → Account → Create New API Token → download `kaggle.json`).

## 1. Environment setup

In [ ]:
!pip install -q timm==1.0.9 ptflops==0.7.4 kagglehub scikit-learn==1.5.2 seaborn opencv-python-headless psutil
import torch, torchvision, timm
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### 1a. Mount Google Drive
Everything this notebook produces — model checkpoints, the best weights for each model, training
history, and all result tables/figures — is written straight to your Drive from here on, **not** to
`/content`. That means a disconnected/recycled free-tier Colab session no longer costs you your
results: just reconnect, re-run from the top, and training below will automatically **skip models
that already finished** and **resume mid-training** for any model that was interrupted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/plant_disease_edge_benchmark"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project files will persist at:", PROJECT_DIR)


In [ ]:
import os, time, json, random, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report, f1_score)
from ptflops import get_model_complexity_info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 15          # raise to 25-30 for the final paper run if time allows
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Everything below lives on Drive so it survives a disconnected Colab session.
RESULTS_DIR = f"{PROJECT_DIR}/results"
CKPT_DIR = f"{RESULTS_DIR}/checkpoints"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Cache the downloaded dataset on Drive too, so you don't re-download ~54k images every session
# (one-time cost of a couple GB of Drive space; every session after the first reuses the cache).
os.environ["KAGGLEHUB_CACHE"] = f"{PROJECT_DIR}/kagglehub_cache"


## 2. Dataset: PlantVillage (Kaggle)
Dataset: `abdallahalidev/plantvillage-dataset` (color images, 38 classes, ~54,305 images).
Get your `kaggle.json` from https://www.kaggle.com/settings → API → Create New Token.

In [ ]:
import kagglehub
# Downloads the dataset once and caches it; no manual kaggle.json juggling needed with kagglehub
dataset_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Dataset cached at:", dataset_path)
!find "{dataset_path}" -maxdepth 3 -type d | head -20


In [ ]:
# Locate the 'color' image folder (ImageFolder-compatible: one sub-folder per class)
import glob
candidates = glob.glob(os.path.join(dataset_path, "**", "color"), recursive=True)
assert candidates, "Could not find the 'color' subset — inspect the tree above and adjust DATA_ROOT manually."
DATA_ROOT = candidates[0]
print("Using:", DATA_ROOT)
print("Num classes:", len(os.listdir(DATA_ROOT)))


In [ ]:
# ---- Transforms ----
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset = datasets.ImageFolder(DATA_ROOT)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes, {len(full_dataset)} images total")

# 70 / 15 / 15 stratified-ish split via random_split (class balance is already good in PlantVillage)
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val
train_idx, val_idx, test_idx = random_split(range(n_total), [n_train, n_val, n_test],
                                              generator=torch.Generator().manual_seed(SEED))

def make_subset(indices, tf):
    ds = datasets.ImageFolder(DATA_ROOT, transform=tf)
    return torch.utils.data.Subset(ds, indices)

train_ds = make_subset(train_idx.indices, train_tf)
val_ds   = make_subset(val_idx.indices, eval_tf)
test_ds  = make_subset(test_idx.indices, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")


## 3. Model zoo
Three architectures, each swapped in as a drop-in classifier head over `NUM_CLASSES`:

| Model | Family | Source |
|---|---|---|
| `mobilenetv3_small` / `mobilenetv3_large` | Lightweight CNN | `torchvision` |
| `efficientnet_lite0` | Lightweight CNN | `timm` |
| `mobilevit_s` | Hybrid CNN+Transformer | `timm` |

All models are initialized from ImageNet-1k pretrained weights and fine-tuned end-to-end (transfer learning — standard practice for edge deployment and for PlantVillage-scale data).

**Why ResNet18 is included:** it is *not* one of the edge-deployment candidates — at ~11.7M
parameters and ~1.8 GFLOPs it is too heavy for typical microcontroller-class edge boards and is included
purely as a **non-lightweight reference point**, so the paper can quantify, in Section IV, exactly how
much accuracy (if any) is sacrificed by moving to the lightweight/hybrid models. Keep it out of any
"best model for deployment" ranking — it exists only for the accuracy-cost comparison.

In [ ]:
def build_model(name, num_classes=NUM_CLASSES):
    if name == "mobilenetv3_small":
        m = torchvision.models.mobilenet_v3_small(weights=torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    elif name == "mobilenetv3_large":
        m = torchvision.models.mobilenet_v3_large(weights=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    elif name == "efficientnet_lite0":
        m = timm.create_model("efficientnet_lite0", pretrained=True, num_classes=num_classes)
    elif name == "mobilevit_s":
        m = timm.create_model("mobilevit_s", pretrained=True, num_classes=num_classes)
    elif name == "ghostnet_100":
        # Pure CNN (ghost modules / cheap linear operations), ~5.2M params — a second efficiency-
        # oriented CNN design philosophy alongside MobileNet/EfficientNet.
        m = timm.create_model("ghostnet_100", pretrained=True, num_classes=num_classes)
    elif name == "edgenext_small":
        # Hybrid CNN-Transformer (like MobileViT, but a different design: split depth-wise
        # transpose-attention blocks). A second, independent hybrid data point.
        m = timm.create_model("edgenext_small", pretrained=True, num_classes=num_classes)
    elif name == "mobilevitv2_100":
        # Hybrid CNN-Transformer: MobileViT's own successor, replacing multi-head self-attention
        # with linear-complexity separable self-attention aimed specifically at mobile latency.
        m = timm.create_model("mobilevitv2_100", pretrained=True, num_classes=num_classes)
    elif name == "resnet18":
        # Non-lightweight reference baseline: how much accuracy does going "lightweight" actually cost?
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        raise ValueError(name)
    return m

# ResNet18 is a standard (non-lightweight) reference point, not a lightweight/edge candidate itself.
# Edge candidates now span two pure CNNs (MobileNetV3 x2), two more CNNs (EfficientNet-Lite0,
# GhostNet), and three independent hybrid CNN-Transformer designs (MobileViT-S, MobileViT-v2,
# EdgeNeXt) — this turns the "CNN vs. hybrid transformer" comparison from a single anecdotal data
# point (one hybrid model) into a genuine multi-model comparison within each family.
MODEL_NAMES = ["mobilenetv3_small", "mobilenetv3_large", "efficientnet_lite0", "mobilevit_s",
               "ghostnet_100", "edgenext_small", "mobilevitv2_100"]
BASELINE_NAME = "resnet18"
ALL_MODEL_NAMES = MODEL_NAMES + [BASELINE_NAME]
for n in ALL_MODEL_NAMES:
    m = build_model(n)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    tag = "  (non-lightweight reference)" if n == BASELINE_NAME else ""
    print(f"{n:22s} -> {n_params:.2f} M parameters{tag}")
    del m


## 4. Training loop (identical protocol for every model — this is what makes the comparison fair)

**Resumability:** every epoch writes a full checkpoint (model, optimizer, scheduler, scaler, best-so-far
weights, and history) to Drive. If Colab disconnects mid-training, just re-run this section — a model
that was mid-training resumes from its last completed epoch, and a model that already finished is
skipped entirely (loaded straight from its saved best weights) rather than retrained.

In [ ]:
def train_one_model(name, epochs=EPOCHS, patience=5):
    ckpt_path = f"{CKPT_DIR}/{name}_ckpt.pt"
    best_path = f"{CKPT_DIR}/{name}_best.pt"
    hist_path = f"{CKPT_DIR}/{name}_history.json"
    done_path = f"{CKPT_DIR}/{name}_done.flag"

    model = build_model(name).to(DEVICE)

    # Already fully trained in a previous session -> skip straight to loading the saved best weights.
    if os.path.exists(done_path) and os.path.exists(best_path):
        print(f"[{name}] already completed in a previous session — loading saved weights, skipping training.")
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        history = json.load(open(hist_path)) if os.path.exists(hist_path) else {}
        return model, history

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    start_epoch = 0
    best_val_acc, epochs_no_improve = 0.0, 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_state = copy.deepcopy(model.state_dict())

    # Mid-training checkpoint from an earlier, disconnected session -> resume exactly where it left off.
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt["best_val_acc"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history = ckpt["history"]
        if os.path.exists(best_path):
            best_state = torch.load(best_path, map_location=DEVICE)
        print(f"[{name}] resuming from epoch {start_epoch} (best_val_acc so far = {best_val_acc:.4f})")

    for epoch in range(start_epoch, epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
        train_loss, train_acc = run_loss/total, correct/total

        model.eval()
        vloss, vcorrect, vtotal = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                loss = criterion(out, y)
                vloss += loss.item() * x.size(0)
                vcorrect += (out.argmax(1) == y).sum().item()
                vtotal += x.size(0)
        val_loss, val_acc = vloss/vtotal, vcorrect/vtotal
        scheduler.step()

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
        print(f"[{name}] epoch {epoch+1:02d}/{epochs}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}  ({time.time()-t0:.0f}s)")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(best_state, best_path)   # persist the new best immediately
        else:
            epochs_no_improve += 1

        # Persist full resumable state to Drive after every single epoch.
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(), "best_val_acc": best_val_acc,
            "epochs_no_improve": epochs_no_improve, "history": history,
        }, ckpt_path)
        with open(hist_path, "w") as f:
            json.dump(history, f)

        if epochs_no_improve >= patience:
            print(f"[{name}] early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    with open(done_path, "w") as f:
        f.write("done")
    return model, history


In [ ]:
# Train all eight models sequentially (~6-8 hours total on a T4).
# Safe to interrupt: re-running this cell after a disconnect resumes in-progress models and
# skips models that already finished, instead of restarting from scratch.
trained_models = {}
for name in ALL_MODEL_NAMES:
    print("="*70, f"\nTraining {name}\n", "="*70)
    model, hist = train_one_model(name)
    trained_models[name] = model


## 5. Accuracy evaluation on the held-out test set

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            out = model(x)
            preds = out.argmax(1).cpu().numpy()
            all_preds.extend(preds); all_labels.extend(y.numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return acc, prec, rec, f1, all_labels, all_preds

eval_rows = []
preds_store = {}
for name, model in trained_models.items():
    acc, prec, rec, f1, y_true, y_pred = evaluate(model, test_loader)
    preds_store[name] = (y_true, y_pred)
    eval_rows.append({"model": name, "test_accuracy": acc, "macro_precision": prec,
                       "macro_recall": rec, "macro_f1": f1})
    print(f"{name:22s} acc={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

eval_df = pd.DataFrame(eval_rows)
eval_df["is_edge_candidate"] = eval_df["model"] != BASELINE_NAME
eval_df.to_csv(f"{RESULTS_DIR}/table_accuracy.csv", index=False)
eval_df


In [ ]:
# Confusion matrices (save one figure per model — use in the paper's Results section)
for name in ALL_MODEL_NAMES:
    y_true, y_pred = preds_store[name]
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, cmap="Blues", cbar=True, xticklabels=False, yticklabels=False)
    plt.title(f"Confusion matrix — {name}")
    plt.xlabel("Predicted class"); plt.ylabel("True class")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/confmat_{name}.png", dpi=200)
    plt.show()


## 6. Efficiency profiling: parameters, FLOPs, model size, CPU latency
This is the edge-deployment section of the paper. Latency is measured **on CPU, single-threaded**,
which is a reasonable proxy for a Raspberry Pi 4 (ARM Cortex-A72, no GPU). If you have physical
access to a Raspberry Pi, re-run the `measure_latency` cell there directly for a real on-device number
and replace the proxy figures — say so explicitly in the paper's Methodology section (see draft).

In [ ]:
def count_params_mb(model):
    n_params = sum(p.numel() for p in model.parameters())
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)
    return n_params, size_mb

def get_flops(model):
    model_cpu = copy.deepcopy(model).to("cpu").eval()
    macs, params = get_model_complexity_info(model_cpu, (3, IMG_SIZE, IMG_SIZE),
                                              as_strings=False, print_per_layer_stat=False, verbose=False)
    flops = 2 * macs  # MACs -> FLOPs
    del model_cpu
    return flops, params

def measure_latency(model, n_warmup=10, n_runs=50, device="cpu", threads=1):
    if device == "cpu":
        torch.set_num_threads(threads)
    m = copy.deepcopy(model).to(device).eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            m(dummy)
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            m(dummy)
            if device == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)  # ms
    del m
    return float(np.mean(times)), float(np.std(times)), float(np.percentile(times, 95))

profile_rows = []
for name, model in trained_models.items():
    n_params, size_mb = count_params_mb(model)
    flops, _ = get_flops(model)
    lat_mean, lat_std, lat_p95 = measure_latency(model, device="cpu", threads=1)
    profile_rows.append({
        "model": name,
        "params_M": n_params / 1e6,
        "model_size_MB": size_mb,
        "GFLOPs": flops / 1e9,
        "cpu_latency_ms_mean": lat_mean,
        "cpu_latency_ms_std": lat_std,
        "cpu_latency_ms_p95": lat_p95,
    })
    print(f"{name:22s} params={n_params/1e6:.2f}M  size={size_mb:.1f}MB  "
          f"GFLOPs={flops/1e9:.3f}  CPU latency={lat_mean:.1f}±{lat_std:.1f} ms")

profile_df = pd.DataFrame(profile_rows)
profile_df["is_edge_candidate"] = profile_df["model"] != BASELINE_NAME
profile_df.to_csv(f"{RESULTS_DIR}/table_efficiency.csv", index=False)
profile_df


In [ ]:
# INT8 post-training quantization for the pure-CNN models via torch.quantization (dynamic quant
# demo). Hybrid CNN-Transformer models (MobileViT-S, MobileViT-v2, EdgeNeXt) are excluded: PyTorch's
# dynamic quantization path does not natively cover their multi-head/separable attention layers, so
# treat this as an optional extra data point for the pure-CNN rows only.
import torch.quantization as tq

QUANT_MODEL_NAMES = ["mobilenetv3_small", "mobilenetv3_large", "efficientnet_lite0",
                      "ghostnet_100", BASELINE_NAME]
quant_rows = []
for name in QUANT_MODEL_NAMES:
    model = copy.deepcopy(trained_models[name]).to("cpu").eval()  # deepcopy! .to() mutates in place otherwise
    q_model = tq.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
    # NB: a dynamically-quantized Linear stores its INT8 weights in packed params that are NOT returned by
    # .parameters(), so count_params_mb() would silently drop them and overstate the size reduction.
    # Serialising the state_dict counts every byte that would actually be shipped.
    import io
    _buf = io.BytesIO(); torch.save(q_model.state_dict(), _buf)
    size_mb = _buf.getbuffer().nbytes / (1024 ** 2)
    lat_mean, lat_std, _ = measure_latency(q_model, device="cpu", threads=1)
    quant_rows.append({"model": name, "quantized_size_MB_linear_only": size_mb,
                        "quantized_cpu_latency_ms": lat_mean})
    print(f"{name:22s} INT8(linear) size={size_mb:.1f}MB  latency={lat_mean:.1f}ms")

pd.DataFrame(quant_rows).to_csv(f"{RESULTS_DIR}/table_quantization_note.csv", index=False)


## 7. Master results table + accuracy/efficiency trade-off plot (this is Table III / Figure 1 of the paper)

In [ ]:
master_df = eval_df.merge(profile_df, on="model")
master_df = master_df.sort_values("test_accuracy", ascending=False).reset_index(drop=True)
master_df.to_csv(f"{RESULTS_DIR}/table_master_results.csv", index=False)
master_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
markers = {"mobilenetv3_small": "o", "mobilenetv3_large": "s", "efficientnet_lite0": "^",
           "mobilevit_s": "D", "resnet18": "*", "ghostnet_100": "v",
           "edgenext_small": "P", "mobilevitv2_100": "X"}
for _, row in master_df.iterrows():
    is_baseline = row["model"] == BASELINE_NAME
    ax.scatter(row["cpu_latency_ms_mean"], row["test_accuracy"]*100,
               s=(row["params_M"]*40 if not is_baseline else 260),
               marker=markers.get(row["model"], "o"),
               color="gray" if is_baseline else None,
               label=row["model"] + (" (reference, not edge-viable)" if is_baseline else ""),
               alpha=0.85, edgecolor="black")
    ax.annotate(row["model"], (row["cpu_latency_ms_mean"], row["test_accuracy"]*100),
                textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.set_xlabel("CPU inference latency (ms, single-image, Raspberry-Pi proxy)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title("Accuracy vs. latency trade-off (bubble size = parameter count)\nResNet18 shown in gray as a non-lightweight reference, not a Pareto candidate")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/tradeoff_plot.png", dpi=220)
plt.show()


## 7b. Bootstrap confidence intervals on test accuracy (free statistical rigor)
Retraining every model over 3–5 random seeds is the gold-standard way to get a confidence interval on
accuracy, but it multiplies your Colab compute budget by 3–5x. As a budget-friendly alternative that
still gives reviewers a real interval rather than a bare point estimate, this cell **bootstrap-resamples
the existing test-set predictions** (already computed in Section 5) 1000 times and reports the 95% CI on
accuracy for every model. This quantifies sampling uncertainty from the finite test set; it does **not**
capture training-run-to-run variance (different random init / data order), which is what multi-seed
retraining would additionally cover — mention this distinction explicitly if you cite the CI in the
paper.

In [ ]:
def bootstrap_accuracy_ci(y_true, y_pred, n_boot=1000, ci=95, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    n = len(y_true)
    boot_accs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_accs[i] = np.mean(y_true[idx] == y_pred[idx])
    lo, hi = np.percentile(boot_accs, [(100-ci)/2, 100-(100-ci)/2])
    return boot_accs.mean(), lo, hi

ci_rows = []
for name in ALL_MODEL_NAMES:
    y_true, y_pred = preds_store[name]
    mean_acc, lo, hi = bootstrap_accuracy_ci(y_true, y_pred)
    ci_rows.append({"model": name, "bootstrap_mean_acc": mean_acc,
                     "ci95_low": lo, "ci95_high": hi})
    print(f"{name:22s} acc={mean_acc*100:.2f}%  95% CI=[{lo*100:.2f}%, {hi*100:.2f}%]")

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(f"{RESULTS_DIR}/table_bootstrap_ci.csv", index=False)
ci_df


## 8. Statistical significance (optional but reviewer-friendly)
A single train/test split does not, on its own, support a claim that model A beats model B.
Run 5-fold cross-validation or 5 repeated seeds if you want a defensible significance claim
(McNemar's test on paired predictions is the standard choice for classifier comparison on one
shared test set). With three independent hybrid CNN-Transformer models now in the comparison
(MobileViT-S, MobileViT-v2, EdgeNeXt) alongside four pure CNNs, the cell below runs McNemar's test
between the single best pure-CNN edge candidate and each of the three hybrids — a "does any
transformer actually beat the best CNN" test repeated three times — and saves all of them to
`table_mcnemar.csv` so they survive the session even if `preds_store` doesn't (it lives in memory
only; if you're re-running just this cell in a later session, re-run Section 5's evaluation cell
first to rebuild `preds_store` — it's fast, just forward passes on the already-trained models, not
retraining).

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(y_true, pred_a, pred_b):
    correct_a = (np.array(pred_a) == np.array(y_true))
    correct_b = (np.array(pred_b) == np.array(y_true))
    n01 = int(np.sum(correct_a & ~correct_b))
    n10 = int(np.sum(~correct_a & correct_b))
    table = [[0, n01], [n10, 0]]
    result = mcnemar(table, exact=(n01 + n10 < 25))
    return result, n01, n10

CNN_NAMES = ["mobilenetv3_small", "mobilenetv3_large", "efficientnet_lite0", "ghostnet_100"]
HYBRID_NAMES = ["mobilevit_s", "mobilevitv2_100", "edgenext_small"]

edge_df = master_df[master_df.model != BASELINE_NAME]  # exclude the non-lightweight reference
best_cnn = (edge_df[edge_df.model.isin(CNN_NAMES)]
            .sort_values("test_accuracy", ascending=False).iloc[0]["model"])
print(f"Best pure-CNN edge candidate: {best_cnn}")

# Best pure-CNN vs. each of the three independent hybrid CNN-Transformer models.
pairs = [(best_cnn, h) for h in HYBRID_NAMES]

mcnemar_rows = []
for a, b in pairs:
    y_true_a, pred_a = preds_store[a]
    y_true_b, pred_b = preds_store[b]
    res, n01, n10 = mcnemar_test(y_true_a, pred_a, pred_b)
    mcnemar_rows.append({
        "model_a": a, "model_b": b, "n_a_correct_b_wrong": n01, "n_b_correct_a_wrong": n10,
        "statistic": res.statistic, "p_value": res.pvalue,
        "significant_at_0.05": res.pvalue < 0.05,
    })
    print(f"McNemar {a:20s} vs {b:18s}: statistic={res.statistic:.3f}  p={res.pvalue:.4f}  "
          f"significant={res.pvalue < 0.05}")

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(f"{RESULTS_DIR}/table_mcnemar.csv", index=False)
mcnemar_df


## 9. Robustness to real-world image perturbations
PlantVillage images are lab-quality: uniform background, consistent lighting, sharp focus. A phone
photo in a field will not look like that. This section simulates that gap cheaply — no retraining —
by re-evaluating every already-trained model on the *same* test set after applying increasing levels
of Gaussian blur, Gaussian noise, and brightness/contrast shift, and plotting how accuracy degrades
for each architecture. A model that degrades gracefully is more trustworthy for real deployment than
one that only looks good on clean lab images.

In [ ]:
import torchvision.transforms.functional as TF

def make_perturbed_loader(severity, kind):
    class PerturbedTestSet(torch.utils.data.Dataset):
        def __init__(self, base_subset, kind, severity):
            self.base = base_subset
            self.kind = kind
            self.severity = severity
        def __len__(self):
            return len(self.base)
        def __getitem__(self, idx):
            x, y = self.base[idx]  # already normalized tensor, C,H,W
            if self.severity == 0 or self.kind == "clean":
                return x, y
            if self.kind == "blur":
                k = [3, 5, 9, 13][self.severity - 1]
                x = TF.gaussian_blur(x, kernel_size=k, sigma=float(self.severity))
            elif self.kind == "noise":
                std = [0.02, 0.05, 0.10, 0.20][self.severity - 1]
                x = x + torch.randn_like(x) * std
            elif self.kind == "brightness":
                factor = [0.85, 0.70, 1.30, 1.55][self.severity - 1] if self.severity <= 4 else 1.0
                # unnormalize -> adjust -> renormalize (approx, cheap and good enough for a robustness probe)
                mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
                std_ = torch.tensor(IMAGENET_STD).view(3, 1, 1)
                x_un = (x * std_ + mean).clamp(0, 1)
                x_un = (x_un * factor).clamp(0, 1)
                x = (x_un - mean) / std_
            return x, y
    return DataLoader(PerturbedTestSet(test_ds, kind, severity), batch_size=BATCH_SIZE,
                       shuffle=False, num_workers=NUM_WORKERS)

PERTURBATIONS = ["blur", "noise", "brightness"]
SEVERITIES = [1, 2, 3, 4]  # clean (severity 0) is computed once per model, separately, below

robustness_rows = []
for name, model in trained_models.items():
    # Clean baseline computed once per model (identical regardless of "kind"), then tagged
    # onto every perturbation type so each of the three plots has a severity-0 anchor point.
    clean_loader = make_perturbed_loader(0, "clean")
    clean_acc, _, _, _, _, _ = evaluate(model, clean_loader)
    for kind in PERTURBATIONS:
        robustness_rows.append({"model": name, "perturbation": kind, "severity": 0, "accuracy": clean_acc})
    for kind in PERTURBATIONS:
        for sev in SEVERITIES:
            loader = make_perturbed_loader(sev, kind)
            acc, _, _, _, _, _ = evaluate(model, loader)
            robustness_rows.append({"model": name, "perturbation": kind, "severity": sev, "accuracy": acc})
    print(f"[{name}] robustness sweep done")

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(f"{RESULTS_DIR}/table_robustness.csv", index=False)
robustness_df.head(10)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, kind in zip(axes, PERTURBATIONS):
    for name in ALL_MODEL_NAMES:
        sub = robustness_df[(robustness_df.model == name) &
                             ((robustness_df.perturbation == kind) | (robustness_df.severity == 0))]
        sub = sub.drop_duplicates(subset="severity").sort_values("severity")
        ax.plot(sub["severity"], sub["accuracy"] * 100, marker="o", label=name)
    ax.set_title(f"{kind} robustness")
    ax.set_xlabel("Severity (0 = clean)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Test accuracy (%)")
axes[-1].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/robustness_plot.png", dpi=200)
plt.show()


## 10. Input-resolution ablation
Edge deployments often deliberately downscale input resolution to cut latency further than
architecture choice alone allows. Since these models use global average pooling, they accept
variable input sizes without retraining — so this section re-evaluates every model's accuracy *and*
re-measures its CPU latency at 96×96, 160×160, and 224×224 (the resolution they were trained at),
giving a second practical lever alongside architecture choice.

In [ ]:
RESOLUTIONS = [96, 160, 224]

def make_resized_loader(res):
    tf = transforms.Compose([
        transforms.Resize((res, res)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    ds = datasets.ImageFolder(DATA_ROOT, transform=tf)
    subset = torch.utils.data.Subset(ds, test_idx.indices)
    return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

def measure_latency_at_res(model, res, device="cpu", n_warmup=10, n_runs=30):
    m = copy.deepcopy(model).to(device).eval()
    dummy = torch.randn(1, 3, res, res).to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            m(dummy)
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            m(dummy)
            times.append((time.perf_counter() - t0) * 1000)
    del m
    return float(np.mean(times))

resolution_rows = []
for res in RESOLUTIONS:
    loader = make_resized_loader(res)
    for name, model in trained_models.items():
        acc, _, _, f1, _, _ = evaluate(model, loader)
        lat = measure_latency_at_res(model, res)
        resolution_rows.append({"model": name, "resolution": res, "accuracy": acc,
                                 "macro_f1": f1, "cpu_latency_ms": lat})
        print(f"[{name}] res={res}: acc={acc:.4f}  f1={f1:.4f}  latency={lat:.1f}ms")

resolution_df = pd.DataFrame(resolution_rows)
resolution_df.to_csv(f"{RESULTS_DIR}/table_resolution_ablation.csv", index=False)
resolution_df


## 11. Grad-CAM: are the models looking at the lesion, or at shortcuts?
PlantVillage's near-100% accuracy is well known to be partly attributable to shortcut learning —
models can sometimes classify from background/pot/lighting cues rather than the actual diseased
tissue, since backgrounds are not perfectly randomized across classes. This section runs a generic,
architecture-agnostic Grad-CAM (hooking the last convolutional layer found in the model) on a handful
of correctly classified sample images per model, so you can visually sanity-check what each model is
actually attending to before trusting the accuracy numbers. Note: for MobileViT-S this hooks the last
convolutional projection before the transformer/pooling stages, which is an approximation, not a
rigorous attention-based explanation — state this caveat explicitly if you use the figure in the paper.

In [ ]:
def find_last_conv(model):
    last_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    return last_conv

def make_gradcam_model(model):
    # Grad-CAM needs to backprop through activations, but several architectures use in-place
    # activations (nn.ReLU(inplace=True) in ResNet/GhostNet, Hardswish/Hardsigmoid in MobileNetV3,
    # SiLU in some hybrid blocks) that overwrite the tensor Grad-CAM's backward hook needs to read,
    # causing "Output 0 of BackwardHookFunctionBackward is a view and is being modified inplace".
    # Fix: work on a deep copy (never mutate the real trained model) and flip *every* module with
    # an `inplace` attribute to False, not just nn.ReLU — Hardswish/Hardsigmoid/SiLU/LeakyReLU/ELU
    # all expose the same attribute and can all trigger the same crash on different architectures.
    m_copy = copy.deepcopy(model)
    for module in m_copy.modules():
        if hasattr(module, "inplace"):
            module.inplace = False
    return m_copy

def grad_cam(model, x, target_class):
    model.eval()
    activations, gradients = {}, {}
    layer = find_last_conv(model)
    def fwd_hook(m, i, o): activations["value"] = o.detach()
    def bwd_hook(m, gi, go): gradients["value"] = go[0].detach()
    h1 = layer.register_forward_hook(fwd_hook)
    h2 = layer.register_full_backward_hook(bwd_hook)

    out = model(x.unsqueeze(0))
    score = out[0, target_class]
    model.zero_grad()
    score.backward()

    h1.remove(); h2.remove()
    acts = activations["value"][0]          # C,H,W
    grads = gradients["value"][0]            # C,H,W
    weights = grads.mean(dim=(1, 2))         # C
    cam = torch.relu((weights[:, None, None] * acts).sum(0))
    cam = cam / (cam.max() + 1e-8)
    return cam.cpu().numpy()

def unnormalize(x):
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std = np.array(IMAGENET_STD).reshape(3, 1, 1)
    img = x.cpu().numpy() * std + mean
    return np.clip(img.transpose(1, 2, 0), 0, 1)

import cv2
N_SAMPLES = 4
sample_indices = random.sample(range(len(test_ds)), N_SAMPLES)

fig, axes = plt.subplots(len(ALL_MODEL_NAMES), N_SAMPLES, figsize=(3*N_SAMPLES, 3*len(ALL_MODEL_NAMES)))
for row, name in enumerate(ALL_MODEL_NAMES):
    gradcam_model = make_gradcam_model(trained_models[name])  # one inplace-safe copy per model
    for col, idx in enumerate(sample_indices):
        x, y = test_ds[idx]
        x = x.to(DEVICE)
        with torch.no_grad():
            pred = gradcam_model(x.unsqueeze(0)).argmax(1).item()
        cam = grad_cam(gradcam_model, x, pred)
        cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        img = unnormalize(x)
        ax = axes[row, col]
        ax.imshow(img)
        ax.imshow(cam_resized, cmap="jet", alpha=0.45)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(name, fontsize=10)
        ax.set_title(f"true={CLASS_NAMES[y][:15]}\npred={CLASS_NAMES[pred][:15]}", fontsize=7)
    del gradcam_model
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/gradcam_grid.png", dpi=180)
plt.show()
print("Inspect gradcam_grid.png: heat should concentrate on lesion/leaf-damage regions, not background.")


## 12. Peak memory footprint during inference
Edge boards (especially microcontroller-class devices) are as RAM-constrained as they are
compute-constrained. This section measures the resident-set size (RSS) added by loading and running
each model — real physical memory the OS hands the process, covering weights, activations, and
PyTorch's internal buffers.

Two earlier attempts at this metric both failed, for two different reasons, and it's worth recording
why: `tracemalloc` only tracks Python-level heap allocations and is blind to PyTorch's C++/ATen
tensor memory, so it silently reported near-zero for every model. Measuring "RSS before vs. after"
*within this same notebook process* also failed, more subtly: by Section 12 the process has already
`deepcopy`'d and loaded every model dozens of times across Sections 5–11's profiling, and Python/
PyTorch's memory allocator does not hand freed pages back to the OS — so the "baseline" RSS captured
right before each model's mini-benchmark was already sitting near the process's all-time peak,
making smaller/earlier-measured models look like they use ~0 MB.

The fix is process isolation: each model is measured in a **brand-new subprocess** that has never
touched any other model, launched fresh via `subprocess.run`, so its RSS baseline is genuinely clean.

In [ ]:
memory_probe_script = f"""
import argparse, sys, psutil, os, torch, torchvision, timm, torch.nn as nn

def build_model(name, num_classes):
    if name == "mobilenetv3_small":
        m = torchvision.models.mobilenet_v3_small(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    elif name == "mobilenetv3_large":
        m = torchvision.models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    elif name == "efficientnet_lite0":
        m = timm.create_model("efficientnet_lite0", pretrained=False, num_classes=num_classes)
    elif name == "mobilevit_s":
        m = timm.create_model("mobilevit_s", pretrained=False, num_classes=num_classes)
    elif name == "ghostnet_100":
        m = timm.create_model("ghostnet_100", pretrained=False, num_classes=num_classes)
    elif name == "edgenext_small":
        m = timm.create_model("edgenext_small", pretrained=False, num_classes=num_classes)
    elif name == "mobilevitv2_100":
        m = timm.create_model("mobilevitv2_100", pretrained=False, num_classes=num_classes)
    elif name == "resnet18":
        m = torchvision.models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        raise ValueError(name)
    return m

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True)
parser.add_argument("--ckpt", required=True)
parser.add_argument("--num_classes", type=int, required=True)
parser.add_argument("--img_size", type=int, required=True)
args = parser.parse_args()

proc = psutil.Process(os.getpid())
baseline_mb = proc.memory_info().rss / (1024 ** 2)   # after imports, before this model exists

model = build_model(args.model, args.num_classes)
model.load_state_dict(torch.load(args.ckpt, map_location="cpu"))
model.eval()

dummy = torch.randn(1, 3, args.img_size, args.img_size)
with torch.no_grad():
    for _ in range(5):
        model(dummy)

peak_mb = proc.memory_info().rss / (1024 ** 2)
print(f"MEMORY_MB:{{max(peak_mb - baseline_mb, 0.0):.4f}}")
"""

script_path = f"{RESULTS_DIR}/_memory_probe.py"
with open(script_path, "w") as f:
    f.write(memory_probe_script)

import subprocess, re

memory_rows = []
for name in ALL_MODEL_NAMES:
    ckpt_path = f"{CKPT_DIR}/{name}_best.pt"
    result = subprocess.run(
        ["python3", script_path, "--model", name, "--ckpt", ckpt_path,
         "--num_classes", str(NUM_CLASSES), "--img_size", str(IMG_SIZE)],
        capture_output=True, text=True,
    )
    match = re.search(r"MEMORY_MB:([\d.]+)", result.stdout)
    if not match:
        print(f"[{name}] probe failed — stderr:\n{result.stderr[-1500:]}")
        peak_mb = float("nan")
    else:
        peak_mb = float(match.group(1))
    memory_rows.append({"model": name, "peak_inference_memory_MB": peak_mb})
    print(f"{name:22s} peak inference memory ≈ {peak_mb:.1f} MB (isolated subprocess)")

memory_df = pd.DataFrame(memory_rows)
memory_df.to_csv(f"{RESULTS_DIR}/table_memory_footprint.csv", index=False)
memory_df


## 13. Part 2: PlantDoc — a harder, real-world benchmark
Every result so far is on PlantVillage: lab photos, uniform background, near-ceiling accuracy for
every architecture (Table I). That's a weak foundation for claiming anything about "real-world"
deployment. This section repeats the same identical training protocol on **PlantDoc** — 2,598 images
across 27 classes, scraped from the open internet, with cluttered backgrounds, variable lighting, and
real photographic noise — to test whether the architecture ranking observed on PlantVillage survives
contact with harder, field-like data.

To avoid any risk of silently overwriting or contaminating Part 1's PlantVillage results (two bugs
earlier in this notebook were caused by exactly this kind of shared mutable state), Part 2 uses
**entirely separate variable names, functions, checkpoint directories, and result directories** —
nothing here reads or writes anything Part 1 produced.

In [ ]:
PLANTDOC_DIR = f"{PROJECT_DIR}/plantdoc_data"
if not os.path.exists(f"{PLANTDOC_DIR}/train"):
    !git clone -q https://github.com/pratikkayal/PlantDoc-Dataset.git "{PLANTDOC_DIR}"
    print("Cloned PlantDoc.")
else:
    print("PlantDoc already present, skipping clone.")

RESULTS_DIR_PD = f"{PROJECT_DIR}/results_plantdoc"
CKPT_DIR_PD = f"{RESULTS_DIR_PD}/checkpoints"
os.makedirs(RESULTS_DIR_PD, exist_ok=True)
os.makedirs(CKPT_DIR_PD, exist_ok=True)

# PlantDoc ships with train/ and test/ only (no val/); we carve a val split out of train/ and leave
# the official test/ split completely untouched, so PlantDoc-reported test accuracy stays comparable
# to numbers other papers report against this same benchmark.
pd_train_pool = datasets.ImageFolder(f"{PLANTDOC_DIR}/train")
CLASS_NAMES_PD = pd_train_pool.classes
NUM_CLASSES_PD = len(CLASS_NAMES_PD)
print(f"PlantDoc: {NUM_CLASSES_PD} classes, {len(pd_train_pool)} train-pool images")

n_total_pd = len(pd_train_pool)
n_val_pd = max(int(0.15 * n_total_pd), NUM_CLASSES_PD)
n_train_pd = n_total_pd - n_val_pd
train_idx_pd, val_idx_pd = random_split(range(n_total_pd), [n_train_pd, n_val_pd],
                                          generator=torch.Generator().manual_seed(SEED))

train_ds_pd = torch.utils.data.Subset(
    datasets.ImageFolder(f"{PLANTDOC_DIR}/train", transform=train_tf), train_idx_pd.indices)
val_ds_pd = torch.utils.data.Subset(
    datasets.ImageFolder(f"{PLANTDOC_DIR}/train", transform=eval_tf), val_idx_pd.indices)
test_ds_pd = datasets.ImageFolder(f"{PLANTDOC_DIR}/test", transform=eval_tf)  # official PlantDoc test split

train_loader_pd = DataLoader(train_ds_pd, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
val_loader_pd = DataLoader(val_ds_pd, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
test_loader_pd = DataLoader(test_ds_pd, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
print(f"train={len(train_ds_pd)}  val={len(val_ds_pd)}  test={len(test_ds_pd)}")


## 14. Train all eight architectures on PlantDoc (identical protocol to Part 1)
A parameterized copy of Part 1's training loop — same optimizer, LR schedule, loss, batch size,
early-stopping patience, and mixed precision — but taking loaders/paths as explicit arguments instead
of reading module-level globals, so it cannot accidentally touch Part 1's `trained_models`,
`RESULTS_DIR`, or checkpoints. Same Drive-backed resumability as Part 1: safe to disconnect and
re-run this cell.

In [ ]:
def train_one_model_v2(name, train_loader, val_loader, num_classes, ckpt_dir,
                        epochs=EPOCHS, patience=5, seed=None):
    # seed=None (default): behaves exactly as before, using whatever RNG state already exists —
    # this is what Part 2's original single-seed run used, and is left untouched for reproducibility.
    # seed=<int>: reseeds torch/numpy/random right before model construction, so a later multi-seed
    # call varies weight init (of the new classifier head), dropout, augmentation, and batch order,
    # while the *data split* (fixed earlier via train_idx/val_idx/test_idx) never changes — this
    # isolates training-run variance from data-split variance, which is the point of a multi-seed run.
    if seed is not None:
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        random.seed(seed); np.random.seed(seed)

    ckpt_path = f"{ckpt_dir}/{name}_ckpt.pt"
    best_path = f"{ckpt_dir}/{name}_best.pt"
    hist_path = f"{ckpt_dir}/{name}_history.json"
    done_path = f"{ckpt_dir}/{name}_done.flag"

    model = build_model(name, num_classes=num_classes).to(DEVICE)

    if os.path.exists(done_path) and os.path.exists(best_path):
        print(f"[{name}] already completed in a previous session — loading saved weights.")
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        history = json.load(open(hist_path)) if os.path.exists(hist_path) else {}
        return model, history

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    start_epoch = 0
    best_val_acc, epochs_no_improve = 0.0, 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_state = copy.deepcopy(model.state_dict())

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt["best_val_acc"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history = ckpt["history"]
        if os.path.exists(best_path):
            best_state = torch.load(best_path, map_location=DEVICE)
        print(f"[{name}] resuming from epoch {start_epoch} (best_val_acc so far = {best_val_acc:.4f})")

    for epoch in range(start_epoch, epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
        train_loss, train_acc = run_loss/total, correct/total

        model.eval()
        vloss, vcorrect, vtotal = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                loss = criterion(out, y)
                vloss += loss.item() * x.size(0)
                vcorrect += (out.argmax(1) == y).sum().item()
                vtotal += x.size(0)
        val_loss, val_acc = vloss/vtotal, vcorrect/vtotal
        scheduler.step()

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
        print(f"[{name}] epoch {epoch+1:02d}/{epochs}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}  ({time.time()-t0:.0f}s)")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(best_state, best_path)
        else:
            epochs_no_improve += 1

        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(), "best_val_acc": best_val_acc,
            "epochs_no_improve": epochs_no_improve, "history": history,
        }, ckpt_path)
        with open(hist_path, "w") as f:
            json.dump(history, f)

        if epochs_no_improve >= patience:
            print(f"[{name}] early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    with open(done_path, "w") as f:
        f.write("done")
    return model, history

trained_models_pd = {}
for name in ALL_MODEL_NAMES:
    print("="*70, f"\nTraining {name} on PlantDoc\n", "="*70)
    model, hist = train_one_model_v2(name, train_loader_pd, val_loader_pd, NUM_CLASSES_PD, CKPT_DIR_PD)
    trained_models_pd[name] = model


## 15. Evaluate on PlantDoc's held-out test set
Note: parameter count, FLOPs, model size, and CPU latency from Table II (Section 5–6) carry over
essentially unchanged — only the final classification layer's output width differs (27 vs. 38
classes), a negligible difference in parameter count that doesn't meaningfully change any efficiency
number. Only accuracy/F1 need to be re-measured here.

In [ ]:
eval_rows_pd = []
preds_store_pd = {}
for name, model in trained_models_pd.items():
    acc, prec, rec, f1, y_true, y_pred = evaluate(model, test_loader_pd)
    preds_store_pd[name] = (y_true, y_pred)
    eval_rows_pd.append({"model": name, "plantdoc_test_accuracy": acc, "plantdoc_macro_precision": prec,
                          "plantdoc_macro_recall": rec, "plantdoc_macro_f1": f1})
    print(f"{name:22s} PlantDoc acc={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

eval_df_pd = pd.DataFrame(eval_rows_pd)
eval_df_pd.to_csv(f"{RESULTS_DIR_PD}/table_accuracy_plantdoc.csv", index=False)
eval_df_pd


In [ ]:
# One representative confusion matrix (highest-accuracy model) — PlantDoc has only ~8-12 test
# images per class, so per-class noise is real; treat this qualitatively, not as a precise per-class
# rate.
best_name_pd = eval_df_pd.sort_values("plantdoc_test_accuracy", ascending=False).iloc[0]["model"]
y_true, y_pred = preds_store_pd[best_name_pd]
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, cmap="Oranges", cbar=True, xticklabels=False, yticklabels=False)
plt.title(f"PlantDoc confusion matrix — {best_name_pd} (best PlantDoc model)")
plt.xlabel("Predicted class"); plt.ylabel("True class")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR_PD}/confmat_plantdoc_{best_name_pd}.png", dpi=200)
plt.show()


## 16. Cross-dataset comparison: does the PlantVillage ranking survive?
This is the key output of Part 2: PlantVillage accuracy (Table I) and PlantDoc accuracy side by side
for every model, plus the generalization gap between them. A large, architecture-dependent gap here
would be direct evidence that the Section IV conclusions ("accuracy is statistically indistinguishable
across architectures") are an artifact of PlantVillage being too easy, not a general truth.

**Important caveat on interpretation:** each model here is trained *and* tested within its own dataset
(PlantVillage-trained models are never evaluated on PlantDoc). The accuracy drop between the two
therefore mixes together harder, uncontrolled imagery, a much smaller training set (~2,000 vs. ~38,000
images), and a different class set (27 vs. 38). It is **not** a pure domain-shift measurement. A
train-on-PlantVillage / test-on-PlantDoc experiment over overlapping classes would be needed to isolate
domain shift from data scarcity.

In [ ]:
cross_df = eval_df.merge(eval_df_pd, on="model")
cross_df["generalization_gap_pp"] = (cross_df["test_accuracy"] - cross_df["plantdoc_test_accuracy"]) * 100
cross_df = cross_df.sort_values("plantdoc_test_accuracy", ascending=False)
cross_df.to_csv(f"{RESULTS_DIR_PD}/table_cross_dataset_comparison.csv", index=False)
print(cross_df[["model", "test_accuracy", "plantdoc_test_accuracy", "generalization_gap_pp"]]
      .to_string(index=False))
cross_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(cross_df))
width = 0.35
ax.bar(x - width/2, cross_df["test_accuracy"] * 100, width, label="PlantVillage (lab)", color="#4C72B0")
ax.bar(x + width/2, cross_df["plantdoc_test_accuracy"] * 100, width, label="PlantDoc (field)", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(cross_df["model"], rotation=25, ha="right")
ax.set_ylabel("Test accuracy (%)")
ax.set_title("Accuracy on each dataset (models trained and tested within their own dataset)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR_PD}/cross_dataset_plot.png", dpi=200)
plt.show()


## 17. Multi-seed retraining, part A: PlantVillage headline models
Retraining all 8 models across multiple seeds on PlantVillage would take 15–20+ hours — not
realistic on free-tier Colab. Instead, this section multi-seeds only the two models whose near-tie
is the paper's headline lab-data finding: **GhostNet-100 and MobileViT-S** (99.865% each on the
original seed-42 run). Two additional seeds (43, 44) are trained here; combined with the existing
seed-42 result, each model gets a real 3-seed accuracy distribution instead of a single point
estimate — this is what actually separates training-run variance from test-set sampling variance
(which the bootstrap CI in Section 7 already covers, but bootstrap alone can't detect if a model
just got a lucky/unlucky weight initialization).

In [ ]:
HEADLINE_MODELS_PV = ["ghostnet_100", "mobilevit_s"]
EXTRA_SEEDS = [43, 44]

multiseed_pv_rows = []
# Seed 42 is already the main run's result — fold it in for a real 3-point distribution.
for name in HEADLINE_MODELS_PV:
    seed42_acc = eval_df.loc[eval_df.model == name, "test_accuracy"].values[0]
    multiseed_pv_rows.append({"model": name, "seed": 42, "test_accuracy": seed42_acc})

for name in HEADLINE_MODELS_PV:
    for seed in EXTRA_SEEDS:
        seed_ckpt_dir = f"{CKPT_DIR}/multiseed_seed{seed}"
        os.makedirs(seed_ckpt_dir, exist_ok=True)
        print("="*70, f"\nTraining {name} (seed={seed}) on PlantVillage\n", "="*70)
        model, _ = train_one_model_v2(name, train_loader, val_loader, NUM_CLASSES,
                                       seed_ckpt_dir, seed=seed)
        acc, _, _, _, _, _ = evaluate(model, test_loader)
        multiseed_pv_rows.append({"model": name, "seed": seed, "test_accuracy": acc})
        print(f"[{name} seed={seed}] test accuracy = {acc:.4f}")
        del model

multiseed_pv_df = pd.DataFrame(multiseed_pv_rows)
multiseed_pv_df.to_csv(f"{RESULTS_DIR}/table_multiseed_plantvillage.csv", index=False)

summary_pv = multiseed_pv_df.groupby("model")["test_accuracy"].agg(["mean", "std", "count"])
summary_pv.columns = ["mean_accuracy", "std_accuracy", "n_seeds"]
print(summary_pv)


In [ ]:
from scipy import stats as scipy_stats

a = multiseed_pv_df[multiseed_pv_df.model == "ghostnet_100"]["test_accuracy"].values
b = multiseed_pv_df[multiseed_pv_df.model == "mobilevit_s"]["test_accuracy"].values
t_stat, p_val = scipy_stats.ttest_ind(a, b, equal_var=False)
print(f"GhostNet-100 (n={len(a)}, mean={a.mean():.5f}, std={a.std():.5f}) vs. "
      f"MobileViT-S (n={len(b)}, mean={b.mean():.5f}, std={b.std():.5f})")
print(f"Welch's t-test: t={t_stat:.3f}, p={p_val:.4f}  "
      f"(low-powered with n=3/side — treat as indicative, not conclusive)")


## 18. Multi-seed retraining, part B: PlantDoc, all eight models
PlantDoc trains fast (~5-10 min/model), so all eight models get the full 3-seed treatment here.
This directly stress-tests the paper's most important new finding: does the "every hybrid beats
every CNN" family-level split from Section 16 hold up once training-run variance is accounted for,
or could it be an artifact of one lucky/unlucky seed per model?

In [ ]:
EXTRA_SEEDS = [43, 44]

multiseed_pd_rows = []
for name in ALL_MODEL_NAMES:
    seed42_acc = eval_df_pd.loc[eval_df_pd.model == name, "plantdoc_test_accuracy"].values[0]
    multiseed_pd_rows.append({"model": name, "seed": 42, "plantdoc_test_accuracy": seed42_acc})

for name in ALL_MODEL_NAMES:
    for seed in EXTRA_SEEDS:
        seed_ckpt_dir = f"{CKPT_DIR_PD}/multiseed_seed{seed}"
        os.makedirs(seed_ckpt_dir, exist_ok=True)
        print("="*70, f"\nTraining {name} (seed={seed}) on PlantDoc\n", "="*70)
        model, _ = train_one_model_v2(name, train_loader_pd, val_loader_pd, NUM_CLASSES_PD,
                                       seed_ckpt_dir, seed=seed)
        acc, _, _, _, _, _ = evaluate(model, test_loader_pd)
        multiseed_pd_rows.append({"model": name, "seed": seed, "plantdoc_test_accuracy": acc})
        print(f"[{name} seed={seed}] PlantDoc test accuracy = {acc:.4f}")
        del model

multiseed_pd_df = pd.DataFrame(multiseed_pd_rows)
multiseed_pd_df.to_csv(f"{RESULTS_DIR_PD}/table_multiseed_plantdoc.csv", index=False)

summary_pd = multiseed_pd_df.groupby("model")["plantdoc_test_accuracy"].agg(["mean", "std", "count"])
summary_pd.columns = ["mean_accuracy", "std_accuracy", "n_seeds"]
summary_pd = summary_pd.sort_values("mean_accuracy", ascending=False)
print(summary_pd)


In [ ]:
# Family-level test: is the hybrid-vs-CNN generalization gap from Section 16 still there once
# each model's accuracy is a 3-seed mean rather than a single point estimate?
HYBRID_NAMES = ["mobilevit_s", "mobilevitv2_100", "edgenext_small"]
CNN_NAMES_ALL = ["mobilenetv3_small", "mobilenetv3_large", "efficientnet_lite0", "ghostnet_100", "resnet18"]

per_model_mean = multiseed_pd_df.groupby("model")["plantdoc_test_accuracy"].mean()
hybrid_means = per_model_mean.loc[HYBRID_NAMES].values
cnn_means = per_model_mean.loc[CNN_NAMES_ALL].values

print("Hybrid family per-model (3-seed) means:", dict(zip(HYBRID_NAMES, hybrid_means.round(4))))
print("CNN family per-model (3-seed) means:   ", dict(zip(CNN_NAMES_ALL, cnn_means.round(4))))

u_stat, p_val = scipy_stats.mannwhitneyu(hybrid_means, cnn_means, alternative="greater")
print(f"\nMann-Whitney U (hybrid > CNN, per-model 3-seed means): U={u_stat:.1f}, p={p_val:.4f}")
print(f"Hybrid family mean: {hybrid_means.mean():.4f}   CNN family mean: {cnn_means.mean():.4f}   "
      f"gap: {(hybrid_means.mean() - cnn_means.mean())*100:.1f} pp")
print("Note: n=3 vs n=5 at the family level is a small sample for a formal hypothesis test — "
      "report the effect size (the gap itself) as the primary evidence and the p-value as "
      "supporting, not load-bearing, given the sample size.")


## 19. Cross-dataset transfer: train on PlantVillage, test on PlantDoc
Sections 13–18 trained and tested each model *within* PlantDoc, so the PlantVillage→PlantDoc accuracy drop mixes
three things that cannot be separated: harder field imagery, a ~19× smaller training set, and a different class set.
This section removes the last two. The **PlantVillage-trained models from Part 1 are applied, unchanged and with no
fine-tuning, to PlantDoc images** whose classes exist in both datasets. If the hybrid advantage seen within PlantDoc
reflects robustness to domain shift, it should reappear here. If it reflects data efficiency on small datasets, it
should not.

**Design.**
* **Evaluation set = every PlantDoc image** (official train + test folders, ~2,600). No PlantDoc image is used for
  training in this experiment, so using all of them is leak-free and shrinks sampling error (~3 pp on the 236-image
  test split → ~1 pp).
* **Class mapping** (`PD2PV` below) matches PlantDoc folder names to PlantVillage classes (same species *and* same
  condition; PlantDoc's "X leaf" classes are treated as healthy). It is verified at run time and the cell stops with
  a clear message if any name does not match. **Please eyeball the sample-image grid** to sanity-check the mapping.
* **Two tiers.** STRICT (primary) drops three mappings whose semantic match is uncertain: *Bell_pepper leaf spot*
  (PlantVillage only has pepper *bacterial* spot), *Corn leaf blight* (PlantDoc mixes northern/southern blight; PlantVillage
  has only northern), and the 2-image *spider mites* class. EXTENDED keeps all mapped classes.
* **Two decision rules.** *Restricted* (primary): argmax over only the PlantVillage classes that exist in the mapping,
  which matches PlantDoc's label space. *38-way*: argmax over all 38 outputs; predicting a PlantVillage class that has no
  PlantDoc counterpart counts as wrong.
* Reported: accuracy with 1,000-resample bootstrap 95% CI, balanced accuracy (mean per-class recall, since PlantDoc is
  imbalanced), the same on the official 236-image test split only, a paired McNemar test (best hybrid vs best CNN), an
  exact family-level test, and 3-seed results for GhostNet-100 and MobileViT-S (the only models with extra PlantVillage seeds).

Runs in ~10–15 min (most of it decoding PlantDoc images once). Needs Sections 1–4 (Part 1 models and `eval_df`) and 13
(`PLANTDOC_DIR`) to have run in this session.

In [ ]:
import re, os
from PIL import Image

def _norm(s): return re.sub(r"[^a-z0-9]", "", s.lower())

# PlantDoc folder name -> PlantVillage class name (same species AND same condition).
PD2PV = {
 "Apple Scab Leaf": "Apple___Apple_scab",
 "Apple leaf": "Apple___healthy",
 "Apple rust leaf": "Apple___Cedar_apple_rust",
 "Bell_pepper leaf": "Pepper,_bell___healthy",
 "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
 "Blueberry leaf": "Blueberry___healthy",
 "Cherry leaf": "Cherry_(including_sour)___healthy",
 "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
 "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
 "Corn rust leaf": "Corn_(maize)___Common_rust_",
 "Peach leaf": "Peach___healthy",
 "Potato leaf early blight": "Potato___Early_blight",
 "Potato leaf late blight": "Potato___Late_blight",
 "Raspberry leaf": "Raspberry___healthy",
 "Soyabean leaf": "Soybean___healthy",
 "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
 "Strawberry leaf": "Strawberry___healthy",
 "Tomato Early blight leaf": "Tomato___Early_blight",
 "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
 "Tomato leaf": "Tomato___healthy",
 "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
 "Tomato leaf late blight": "Tomato___Late_blight",
 "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
 "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
 "Tomato mold leaf": "Tomato___Leaf_Mold",
 "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
 "grape leaf": "Grape___healthy",
 "grape leaf black rot": "Grape___Black_rot",
}
UNCERTAIN_PD = {"Bell_pepper leaf spot", "Corn leaf blight", "Tomato two spotted spider mites leaf"}

pv_norm = {_norm(n): i for i, n in enumerate(CLASS_NAMES)}        # PlantVillage class names from Part 1
pd_norm_map = {_norm(k): (k, v) for k, v in PD2PV.items()}

pd_items = []                                                      # (path, plantdoc folder, split)
for split in ["train", "test"]:
    root = f"{PLANTDOC_DIR}/{split}"
    for cls in sorted(os.listdir(root)):
        cdir = os.path.join(root, cls)
        if not os.path.isdir(cdir): continue
        for fn in sorted(os.listdir(cdir)):
            if fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                pd_items.append((os.path.join(cdir, fn), cls, split))

found = sorted({c for _, c, _ in pd_items})
unmapped = [c for c in found if _norm(c) not in pd_norm_map]
missing_pv = [pd_norm_map[_norm(c)][1] for c in found
              if _norm(c) in pd_norm_map and _norm(pd_norm_map[_norm(c)][1]) not in pv_norm]
print(f"PlantDoc class folders found: {len(found)} | images (train+test): {len(pd_items)}")
if unmapped or missing_pv:
    print("PlantDoc folders with NO entry in PD2PV:", unmapped)
    print("PD2PV targets that match NO PlantVillage class:", missing_pv)
    print("PlantVillage classes are:", CLASS_NAMES)
    raise ValueError("Fix PD2PV so every name matches, then re-run this cell (nothing was evaluated).")

pd_paths = [p for p, _, _ in pd_items]
pd_cls   = np.array([c for _, c, _ in pd_items])
pd_split = np.array([s for _, _, s in pd_items])
pv_label = np.array([pv_norm[_norm(pd_norm_map[_norm(c)][1])] for c in pd_cls])
is_uncertain = np.array([pd_norm_map[_norm(c)][0] in UNCERTAIN_PD for c in pd_cls])
TIERS = {"strict": ~is_uncertain, "extended": np.ones(len(pd_cls), dtype=bool)}
TIER_COLS = {t: sorted(set(pv_label[m].tolist())) for t, m in TIERS.items()}

print(f"\nSTRICT: {TIERS['strict'].sum()} images, {len(TIER_COLS['strict'])} classes | "
      f"EXTENDED: {TIERS['extended'].sum()} images, {len(TIER_COLS['extended'])} classes\n")
print(f"{'PlantDoc folder':40s} -> {'PlantVillage class':52s} {'n':>5s}  note")
for c in found:
    k, v = pd_norm_map[_norm(c)]
    print(f"{c:40s} -> {CLASS_NAMES[pv_norm[_norm(v)]]:52s} {int((pd_cls == c).sum()):5d}  {'UNCERTAIN (excluded from STRICT)' if k in UNCERTAIN_PD else ''}")


In [ ]:
# Sanity-check the mapping visually: one random PlantDoc image per class, titled with its mapped PlantVillage class.
import random as _random
_rng = _random.Random(0)
fig, axes = plt.subplots(5, 6, figsize=(16, 14))
for ax, c in zip(axes.ravel(), found):
    p = _rng.choice([pth for pth, cc in zip(pd_paths, pd_cls) if cc == c])
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
    ax.set_title(f"{c}\n-> {CLASS_NAMES[pv_norm[_norm(pd_norm_map[_norm(c)][1])]][:34]}", fontsize=7)
for ax in axes.ravel()[len(found):]: ax.axis("off")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/transfer_mapping_check.png", dpi=110); plt.show()


In [ ]:
# Decode every PlantDoc image ONCE (resize to 224x224 exactly like eval_tf) and cache as uint8, so each model is
# evaluated from memory (~0.4 GB) instead of re-decoding thousands of large JPEGs per model.
_resize = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.PILToTensor()])
class _PathSet(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return _resize(Image.open(self.paths[i]).convert("RGB"))
_loader = DataLoader(_PathSet(pd_paths), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
X_pd = torch.cat([b for b in _loader])
print("cached:", tuple(X_pd.shape), X_pd.dtype, f"~{X_pd.numel()/1e6:.0f} MB")


In [ ]:
from scipy import stats as scipy_stats
HYBRID_SET = {"mobilevit_s", "mobilevitv2_100", "edgenext_small"}
MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)

@torch.no_grad()
def pd_logits(model, bs=64):
    model = model.to(DEVICE).eval()
    out = []
    for i in range(0, len(X_pd), bs):
        xb = X_pd[i:i + bs].to(DEVICE).float().div_(255.0)          # identical to ToTensor()
        out.append(model((xb - MEAN_T) / STD_T).float().cpu())       # identical to Normalize()
    return torch.cat(out).numpy()

def transfer_metrics(logits, tier, split=None, boot=0, seed=0):
    m = TIERS[tier].copy()
    if split is not None: m &= (pd_split == split)
    cols = np.array(TIER_COLS[tier])
    lg, y, cls = logits[m], pv_label[m], pd_cls[m]
    pred38 = lg.argmax(1)                                   # 38-way decision
    predR = cols[lg[:, cols].argmax(1)]                     # restricted to classes present in the mapping
    c38, cR = (pred38 == y), (predR == y)
    per_cls = pd.Series(cR.astype(float)).groupby(cls).mean()
    out = dict(n=int(m.sum()), acc_restricted=float(cR.mean()), acc_38way=float(c38.mean()),
               balanced_acc=float(per_cls.mean()))
    if boot:
        idx = np.random.default_rng(seed).integers(0, len(cR), (boot, len(cR)))
        out["ci_lo"], out["ci_hi"] = np.percentile(cR[idx].mean(1), [2.5, 97.5])
    return out, cR, per_cls

rows, correct_strict, percls_all = [], {}, {}
for name in ALL_MODEL_NAMES:
    lg = pd_logits(trained_models[name])
    s, cR, pc = transfer_metrics(lg, "strict", boot=1000)
    e, _, _   = transfer_metrics(lg, "extended")
    t, _, _   = transfer_metrics(lg, "strict", split="test")
    correct_strict[name], percls_all[name] = cR, pc
    rows.append({"model": name, "family": "hybrid" if name in HYBRID_SET else "cnn",
        "plantvillage_acc": float(eval_df.loc[eval_df.model == name, "test_accuracy"].values[0]),
        "transfer_acc_strict": s["acc_restricted"], "ci_lo": s["ci_lo"], "ci_hi": s["ci_hi"],
        "transfer_acc_strict_38way": s["acc_38way"], "transfer_balanced_acc_strict": s["balanced_acc"],
        "transfer_acc_extended": e["acc_restricted"], "transfer_acc_strict_testsplit_only": t["acc_restricted"],
        "n_strict": s["n"], "n_extended": e["n"], "n_testsplit": t["n"]})
    print(f"{name:20s} strict-restricted {s['acc_restricted']*100:5.1f}% [{s['ci_lo']*100:4.1f}-{s['ci_hi']*100:4.1f}]  "
          f"38-way {s['acc_38way']*100:5.1f}%  balanced {s['balanced_acc']*100:5.1f}%  extended {e['acc_restricted']*100:5.1f}%")

transfer_df = pd.DataFrame(rows).sort_values("transfer_acc_strict", ascending=False).reset_index(drop=True)
# Context only (NOT directly comparable: different label sets / test images): PlantDoc-trained, 3-run mean over all 27 classes
try:
    _w = multiseed_pd_df.groupby("model")["plantdoc_test_accuracy"].mean()
    transfer_df["plantdoc_trained_within_mean_3runs"] = transfer_df.model.map(_w)
except NameError:
    pass
transfer_df.to_csv(f"{RESULTS_DIR}/table_transfer_pv_to_pd.csv", index=False)
pd.DataFrame(percls_all).to_csv(f"{RESULTS_DIR}/table_transfer_per_class.csv")
print(); print(transfer_df.round(4).to_string(index=False))


In [ ]:
# ---- family-level and paired tests (original run of each model) ----
H = transfer_df[transfer_df.family == "hybrid"].transfer_acc_strict.values
C = transfer_df[transfer_df.family == "cnn"].transfer_acc_strict.values
u, p = scipy_stats.mannwhitneyu(H, C, alternative="greater")
print(f"Hybrid mean {H.mean()*100:.1f}% vs CNN mean {C.mean()*100:.1f}% (gap {100*(H.mean()-C.mean()):+.1f} pp) | "
      f"one-sided Mann-Whitney U={u:.0f}, p={p:.4f}  (smallest possible with 3 vs 5 models: 0.0179)")
ranks = transfer_df.model.tolist()
print("Rank order:", " > ".join(ranks)); print("All three hybrids in top 3:", set(ranks[:3]) == HYBRID_SET)

best_h = transfer_df[transfer_df.family == "hybrid"].iloc[0].model
best_c = transfer_df[transfer_df.family == "cnn"].iloc[0].model
a, b = correct_strict[best_h], correct_strict[best_c]
n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
from statsmodels.stats.contingency_tables import mcnemar
res = mcnemar([[0, n01], [n10, 0]], exact=(n01 + n10 < 25))
idx = np.random.default_rng(1).integers(0, len(a), (2000, len(a)))
dlo, dhi = np.percentile(a[idx].mean(1) - b[idx].mean(1), [2.5, 97.5])
print(f"\nPaired: {best_h} vs {best_c}: acc diff {100*(a.mean()-b.mean()):+.1f} pp (95% CI {100*dlo:+.1f}..{100*dhi:+.1f}) "
      f"| McNemar p={res.pvalue:.4g} (discordant {n01}/{n10})")

# ---- extra seeds for the two models that have them ----
ms_rows = []
for name in ["ghostnet_100", "mobilevit_s"]:
    accs = [float(transfer_df.set_index("model").loc[name, "transfer_acc_strict"])]
    for seed in [43, 44]:
        path = f"{CKPT_DIR}/multiseed_seed{seed}/{name}_best.pt"
        if not os.path.exists(path):
            print("missing checkpoint:", path); continue
        m = build_model(name, num_classes=NUM_CLASSES).to(DEVICE)
        m.load_state_dict(torch.load(path, map_location=DEVICE))
        accs.append(transfer_metrics(pd_logits(m), "strict")[0]["acc_restricted"]); del m
    ms_rows.append({"model": name, "n_runs": len(accs), "runs_pct": [round(x*100, 2) for x in accs],
                    "mean_pct": np.mean(accs)*100, "sd_pct": (np.std(accs, ddof=1)*100 if len(accs) > 1 else np.nan)})
transfer_ms_df = pd.DataFrame(ms_rows)
transfer_ms_df.to_csv(f"{RESULTS_DIR}/table_transfer_multiseed.csv", index=False)
print(); print(transfer_ms_df.round(2).to_string(index=False))

# ---- figure ----
d = transfer_df
fig, ax = plt.subplots(figsize=(8.4, 4.8)); x = np.arange(len(d)); w = 0.38
ax.bar(x - w/2, d.plantvillage_acc*100, w, color="#4C72B0")
cols_ = ["#C44E52" if f == "hybrid" else "#DD8452" for f in d.family]
ax.bar(x + w/2, d.transfer_acc_strict*100, w, color=cols_,
       yerr=[(d.transfer_acc_strict - d.ci_lo)*100, (d.ci_hi - d.transfer_acc_strict)*100], capsize=3, error_kw=dict(lw=1))
for i, v in enumerate(d.transfer_acc_strict*100): ax.text(i + w/2, v + 3.5, f"{v:.1f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(d.model, rotation=25, ha="right", fontsize=8.5)
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 125)
ax.set_title("Train on PlantVillage, test on PlantDoc (no fine-tuning; STRICT classes, restricted decision; 95% bootstrap CI)", fontsize=8.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#4C72B0", label="PlantVillage test (in-domain)"), Patch(color="#C44E52", label="PlantDoc transfer: hybrid"),
                   Patch(color="#DD8452", label="PlantDoc transfer: CNN")], loc="upper right", fontsize=8, ncol=3)
ax.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/transfer_plot.png", dpi=200); plt.show()


## 20. Full static INT8 quantization via ONNX Runtime
Section 6's dynamic quantization was a deliberately honest negative result: it barely touched latency
because it only quantizes `nn.Linear` layers and these architectures are convolution-dominated. This
section applies **static** INT8 quantization (weights *and* activations, per-channel weights, calibrated
on 128 real PlantVillage training images) to every model's ONNX export, then measures accuracy, file
size, and single-thread CPU latency with the **same runtime** (ONNX Runtime) for both the FP32 and INT8
versions, so the comparison is apples-to-apples.

**Why ONNX Runtime and not TFLite:** the ONNX → TensorFlow → TFLite chain depends on a TensorFlow
build whose protobuf requirement clashes with Colab's preinstalled protobuf (`VersionError: gencode
7.35.1 runtime 6.33.6`). ONNX Runtime needs neither TensorFlow nor a protobuf upgrade, and it is itself a
production edge runtime (it runs on ARM/Raspberry Pi).

**Scope note:** all eight models are attempted. Static quantization of the hybrid models' attention
layers is less mature than for CNNs, so a hybrid may fail to export/quantize or may lose accuracy; each
model is wrapped in try/except and reported as-is rather than hacked around. Accuracy is evaluated on the
same fixed random subset of 2,000 PlantVillage test images for FP32 and INT8 (paired comparison).
Do **not** run the old TFLite cells.

In [ ]:
!pip install -q onnx onnxruntime
import onnxruntime as ort
print("onnxruntime", ort.__version__)


In [ ]:
import copy, os, random, time
import numpy as np
from onnxruntime.quantization import (quantize_static, CalibrationDataReader, QuantType,
                                       QuantFormat, CalibrationMethod)
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
except Exception:
    quant_pre_process = None

ORT_DIR = "/content/onnx_int8"          # local scratch: .onnx files are large, keep them off Drive/exports
os.makedirs(ORT_DIR, exist_ok=True)

# Calibration images: real training-split images with the *evaluation* transform (no random augmentation).
_calib_base = datasets.ImageFolder(DATA_ROOT, transform=eval_tf)
_calib_pool = torch.utils.data.Subset(_calib_base, train_idx.indices)
_calib_ids = random.Random(0).sample(range(len(_calib_pool)), 128)
calib_arrays = [_calib_pool[i][0].unsqueeze(0).numpy().astype(np.float32) for i in _calib_ids]

class CalibReader(CalibrationDataReader):
    def __init__(self, input_name, arrays):
        self._it = iter([{input_name: a} for a in arrays])
    def get_next(self):
        return next(self._it, None)

def export_onnx(name, model, path):
    m = copy.deepcopy(model).to("cpu").eval()      # deepcopy: .to() mutates in place otherwise
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    try:
        torch.onnx.export(m, (dummy,), path, input_names=["input"], output_names=["output"],
                           opset_version=18)
    except Exception as e:
        print(f"[{name}] default exporter failed ({type(e).__name__}); retrying with TorchScript exporter")
        torch.onnx.export(m, (dummy,), path, input_names=["input"], output_names=["output"],
                           opset_version=17, dynamo=False)

onnx_paths = {}          # name -> (fp32_path, int8_path)
export_status = {}
for name in ALL_MODEL_NAMES:
    fp32_path = f"{ORT_DIR}/{name}_fp32.onnx"
    prep_path = f"{ORT_DIR}/{name}_prep.onnx"
    int8_path = f"{ORT_DIR}/{name}_int8.onnx"
    try:
        export_onnx(name, trained_models[name], fp32_path)
        src = fp32_path
        if quant_pre_process is not None:
            try:
                quant_pre_process(fp32_path, prep_path)
                src = prep_path
            except Exception as e:
                print(f"[{name}] quant_pre_process skipped ({type(e).__name__}); quantizing raw export")
        quantize_static(
            src, int8_path, CalibReader("input", calib_arrays),
            quant_format=QuantFormat.QDQ, per_channel=True,
            activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
            calibrate_method=CalibrationMethod.MinMax)
        onnx_paths[name] = (fp32_path, int8_path)
        export_status[name] = "ok"
        print(f"[{name}] exported + quantized  fp32={os.path.getsize(fp32_path)/1024**2:.2f}MB "
              f"int8={os.path.getsize(int8_path)/1024**2:.2f}MB")
    except Exception as e:
        export_status[name] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
        print(f"[{name}] FAILED — {export_status[name]}")
print()
print({k: v for k, v in export_status.items()})


In [ ]:
HYBRIDS = {"mobilevit_s", "mobilevitv2_100", "edgenext_small"}
N_EVAL = 2000
_eval_ids = np.random.default_rng(0).choice(len(test_ds), size=N_EVAL, replace=False)
_eval_x = [test_ds[int(i)][0].unsqueeze(0).numpy().astype(np.float32) for i in _eval_ids]
_eval_y = np.array([test_ds[int(i)][1] for i in _eval_ids])

def make_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1; so.inter_op_num_threads = 1      # single thread, Raspberry-Pi-style
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

def ort_predict(sess):
    inp = sess.get_inputs()[0].name
    return np.array([int(np.argmax(sess.run(None, {inp: x})[0])) for x in _eval_x])

def ort_latency(sess, n_warmup=10, n_runs=50):
    inp = sess.get_inputs()[0].name
    x = _eval_x[0]
    for _ in range(n_warmup): sess.run(None, {inp: x})
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); sess.run(None, {inp: x}); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(ts)), float(np.std(ts))

rows = []
for name, (fp32_path, int8_path) in onnx_paths.items():
    s32, s8 = make_session(fp32_path), make_session(int8_path)
    p32, p8 = ort_predict(s32), ort_predict(s8)
    l32, l32s = ort_latency(s32); l8, l8s = ort_latency(s8)
    sz32, sz8 = os.path.getsize(fp32_path) / 1024**2, os.path.getsize(int8_path) / 1024**2
    rows.append({
        "model": name, "family": "hybrid" if name in HYBRIDS else "cnn",
        "fp32_acc": (p32 == _eval_y).mean(), "int8_acc": (p8 == _eval_y).mean(),
        "acc_drop_pp": ((p32 == _eval_y).mean() - (p8 == _eval_y).mean()) * 100,
        "pred_agreement_pct": (p32 == p8).mean() * 100,
        "fp32_size_MB": sz32, "int8_size_MB": sz8, "size_reduction_pct": (1 - sz8 / sz32) * 100,
        "fp32_latency_ms": l32, "int8_latency_ms": l8, "speedup_x": l32 / l8, "n_eval": N_EVAL})
    print(f"{name:20s} acc {rows[-1]['fp32_acc']:.4f} -> {rows[-1]['int8_acc']:.4f}  "
          f"size {sz32:.1f} -> {sz8:.1f} MB  latency {l32:.1f} -> {l8:.1f} ms ({rows[-1]['speedup_x']:.2f}x)")

ort_df = pd.DataFrame(rows)
ort_df.to_csv(f"{RESULTS_DIR}/table_static_int8_onnxruntime.csv", index=False)
print()
print(ort_df.round(3).to_string(index=False))
for k, v in export_status.items():
    if v != "ok": print(f"NOT INCLUDED: {k} -> {v}")


## 21. Zip a lightweight results-only copy for download
Your full checkpoints (`*_ckpt.pt`, needed only for resuming interrupted training) already total
hundreds of MB per part and are **not needed for analysis or sharing** — they stay safely on Drive
regardless of this cell. This export deliberately **excludes** them and includes only what's actually
needed to analyze results or write the paper: CSVs, PNGs, and training-history JSONs. This keeps the
shareable zip in the low tens of MB instead of 500MB+.

If you specifically need the trained weights themselves (e.g., to actually deploy a model), grab the
individual `*_best.pt` file(s) you need directly from Drive — no need to zip everything.

In [ ]:
import shutil, glob

export_dir = f"{PROJECT_DIR}/_export_staging"
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)

# Copy each part's results, but only non-checkpoint files (tables, figures, history) — skip the
# large *_ckpt.pt / *_best.pt weight files entirely, keeping the export small and shareable.
def copy_results_only(src_root, dst_root):
    for dirpath, dirnames, filenames in os.walk(src_root):
        rel = os.path.relpath(dirpath, src_root)
        for fname in filenames:
            if fname.endswith("_ckpt.pt") or fname.endswith("_best.pt"):
                continue  # large training-resume / weight files — not needed for analysis
            src_path = os.path.join(dirpath, fname)
            dst_dir = os.path.join(dst_root, rel)
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src_path, os.path.join(dst_dir, fname))

copy_results_only(f"{PROJECT_DIR}/results", f"{export_dir}/results")
if os.path.exists(f"{PROJECT_DIR}/results_plantdoc"):
    copy_results_only(f"{PROJECT_DIR}/results_plantdoc", f"{export_dir}/results_plantdoc")
else:
    print("Note: results_plantdoc/ not found yet — Part 2 hasn't been run in this session. "
          "Exporting Part 1 only.")

zip_path = shutil.make_archive(f"{PROJECT_DIR}/results_export", "zip", export_dir)
shutil.rmtree(export_dir)
size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"Zipped (results only, no checkpoints) to Drive at: {zip_path}  ({size_mb:.1f} MB)")
from google.colab import files
files.download(zip_path)
